# Preparación de ubicaciones candidatas para expansión de ECOBICI

## Objetivo

Construir el conjunto de ubicaciones candidatas para instalar nuevas estaciones ECOBICI, utilizando una cuadrícula de 250 × 250 metros sobre la zona urbana de CDMX.

Se identificarán las cuadrículas que contienen estaciones existentes y se calcularán las mismas nueve variables utilizadas durante el entrenamiento de los modelos predictivos, tomando como referencia el centro de cada cuadrícula.

Posteriormente, se estimará la demanda potencial mediante Ridge polinómica y Random Forest, y se evaluará la aplicabilidad de ambos modelos a las nuevas ubicaciones.

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [ ]:
cuadriculas = gpd.read_file(
    "../Data/Procesados/cuadriculas_candidatas.gpkg"
)

estaciones = pd.read_csv(
    "../Data/Procesados/variables_estaciones_ECOBICI.csv"
)

estaciones_geo = gpd.GeoDataFrame(
    estaciones,
    geometry=gpd.points_from_xy(
        estaciones["longitud"],
        estaciones["latitud"]
    ),
    crs="EPSG:4326"
).to_crs(cuadriculas.crs)

print("Cuadrículas:", len(cuadriculas))
print("Estaciones:", len(estaciones_geo))
print("CRS:", cuadriculas.crs)

Cuadrículas: 13543
Estaciones: 677
CRS: EPSG:32614


In [4]:
cruces = gpd.sjoin(
    estaciones_geo[["num_cicloe", "geometry"]],
    cuadriculas[["id_celda", "geometry"]],
    how="inner",
    predicate="intersects"
)

celdas_ocupadas = cruces["id_celda"].unique()

candidatas = cuadriculas[
    ~cuadriculas["id_celda"].isin(celdas_ocupadas)
].copy()

print("Estaciones localizadas en cuadrículas:",
      cruces["num_cicloe"].nunique())

print("Cuadrículas ocupadas:", len(celdas_ocupadas))
print("Cuadrículas disponibles:", len(candidatas))

Estaciones localizadas en cuadrículas: 677
Cuadrículas ocupadas: 589
Cuadrículas disponibles: 12954


In [5]:
puntos_candidatos = candidatas[["id_celda", "geometry"]].copy()

# Representar cada cuadrícula por su centro
puntos_candidatos["geometry"] = puntos_candidatos.geometry.centroid

print("Puntos candidatos:", len(puntos_candidatos))
print("CRS:", puntos_candidatos.crs)

puntos_candidatos.head()

Puntos candidatos: 12954
CRS: EPSG:32614


,id_celda,geometry
0,82,POINT (463875 2135125)
1,282,POINT (464125 2134375)
2,283,POINT (464125 2134625)
3,284,POINT (464125 2134875)
4,285,POINT (464125 2135125)


In [6]:
cercanas = gpd.sjoin_nearest(
    puntos_candidatos,
    estaciones_geo[["geometry"]],
    how="left",
    distance_col="dist_ecobici_min"
)

distancias = cercanas.groupby("id_celda")["dist_ecobici_min"].min()

puntos_candidatos["dist_ecobici_min"] = (
    puntos_candidatos["id_celda"].map(distancias)
)

print(puntos_candidatos["dist_ecobici_min"].describe())

print("\nA menos de 250 m de ECOBICI:",
      (puntos_candidatos["dist_ecobici_min"] < 250).sum())

print("A más de 500 m de ECOBICI:",
      (puntos_candidatos["dist_ecobici_min"] > 500).sum())

count    12954.000000
mean      8195.978479
std       5778.755134
min        126.310963
25%       3580.527773
50%       7353.298634
75%      11914.872982
max      25769.690032
Name: dist_ecobici_min, dtype: float64

A menos de 250 m de ECOBICI: 520
A más de 500 m de ECOBICI: 12140


In [7]:
buffers_candidatos = puntos_candidatos[["id_celda", "geometry"]].copy()

buffers_candidatos["geometry"] = (
    buffers_candidatos.geometry.buffer(500)
)

print("Buffers creados:", len(buffers_candidatos))

Buffers creados: 12954


In [8]:
cruces_ecobici = gpd.sjoin(
    estaciones_geo[["num_cicloe", "geometry"]],
    buffers_candidatos,
    how="inner",
    predicate="within"
)

conteos = cruces_ecobici.groupby("id_celda")["num_cicloe"].nunique()

puntos_candidatos["estaciones_ecobici_500m"] = (
    puntos_candidatos["id_celda"].map(conteos).fillna(0).astype(int)
)

print(puntos_candidatos["estaciones_ecobici_500m"].describe())

print(
    "\nCuadrículas sin estaciones ECOBICI a 500 m:",
    (puntos_candidatos["estaciones_ecobici_500m"] == 0).sum()
)

count    12954.000000
mean         0.307627
std          1.396334
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         19.000000
Name: estaciones_ecobici_500m, dtype: float64

Cuadrículas sin estaciones ECOBICI a 500 m: 12141


In [9]:
from shapely.geometry import Point

metro = gpd.read_file(
    "../Data/Info_transporte/stcmetro_shp/STC_Metro_estaciones_utm14n.shp"
).to_crs(epsg=32614)

metrobus = gpd.read_file(
    "../Data/Info_transporte/mb_shp/Metrobus_estaciones.shp"
)

# Mantener consistencia con el entrenamiento
metrobus = metrobus[
    metrobus["TIPO"] != "Servicio temporal"
].to_crs(epsg=32614)

print("Metro:", len(metro))
print("Metrobús:", len(metrobus))

Metro: 195
Metrobús: 307


In [10]:
for nombre, transporte in [
    ("dist_metro", metro),
    ("dist_metrobus", metrobus)
]:
    cercanos = gpd.sjoin_nearest(
        puntos_candidatos[["id_celda", "geometry"]],
        transporte[["geometry"]],
        how="left",
        distance_col=nombre
    )

    puntos_candidatos[nombre] = (
        puntos_candidatos["id_celda"]
        .map(cercanos.groupby("id_celda")[nombre].min())
    )

# Distancia al Zócalo
centro = gpd.GeoSeries(
    [Point(-99.1332, 19.4326)],
    crs="EPSG:4326"
).to_crs(epsg=32614)

puntos_candidatos["dist_centro"] = (
    puntos_candidatos.geometry.distance(centro.iloc[0])
)

print(
    puntos_candidatos[
        ["dist_metro", "dist_metrobus", "dist_centro"]
    ].describe().round(2)
)

       dist_metro  dist_metrobus  dist_centro
count    12954.00       12954.00     12954.00
mean      4304.79        4433.97     14507.96
std       4110.82        4131.37      7088.78
min         10.56           8.14       160.98
25%        915.75        1108.92      8677.13
50%       2583.63        3035.71     14081.40
75%       7095.83        6769.08     19707.12
max      22109.50       19039.18     34424.64


In [11]:
# Cargar población por manzana
poblacion = gpd.read_file(
    "../Data/Procesados/poblacion_geo.gpkg"
)[["POBTOT_num", "geometry"]].to_crs(puntos_candidatos.crs)

# Representar cada manzana mediante un punto interior
poblacion["geometry"] = poblacion.geometry.representative_point()

# Identificar manzanas dentro de cada buffer de 500 m
cruces_pob = gpd.sjoin(
    poblacion,
    buffers_candidatos[["id_celda", "geometry"]],
    how="inner",
    predicate="within"
)

# Sumar población por cuadrícula
pob_por_celda = cruces_pob.groupby("id_celda")["POBTOT_num"].sum()

puntos_candidatos["poblacion_500m"] = (
    puntos_candidatos["id_celda"].map(pob_por_celda).fillna(0)
)

print(puntos_candidatos["poblacion_500m"].describe().round(2))
print(
    "Cuadrículas sin población registrada:",
    (puntos_candidatos["poblacion_500m"] == 0).sum()
)

count    12954.00
mean      8286.48
std       5983.79
min          0.00
25%       3178.00
50%       7365.00
75%      12532.25
max      36245.00
Name: poblacion_500m, dtype: float64
Cuadrículas sin población registrada: 212


In [12]:
denue = pd.read_csv(
    "../Data/INEGI/denue_inegi_09_.csv",
    encoding="latin-1",
    usecols=["latitud", "longitud", "per_ocu", "codigo_act"]
)

personal_aprox = {
    "0 a 5 personas": 2.5,
    "6 a 10 personas": 8,
    "11 a 30 personas": 20.5,
    "31 a 50 personas": 40.5,
    "51 a 100 personas": 75.5,
    "101 a 250 personas": 175.5,
    "251 y más personas": 251
}

denue["personal_aprox"] = denue["per_ocu"].map(personal_aprox)

denue["universidad"] = (
    pd.to_numeric(denue["codigo_act"], errors="coerce")
    .isin([611311, 611312])
    .astype(int)
)

denue_geo = gpd.GeoDataFrame(
    denue,
    geometry=gpd.points_from_xy(denue.longitud, denue.latitud),
    crs="EPSG:4326"
).to_crs(puntos_candidatos.crs)

print("Establecimientos:", len(denue_geo))
print("Personal sin clasificar:", denue_geo["personal_aprox"].isna().sum())

Establecimientos: 460762
Personal sin clasificar: 0


In [13]:
cruces_denue = gpd.sjoin(
    denue_geo[["personal_aprox", "universidad", "geometry"]],
    buffers_candidatos[["id_celda", "geometry"]],
    how="inner",
    predicate="within"
)

resumen_denue = cruces_denue.groupby("id_celda").agg(
    negocios_500m=("personal_aprox", "size"),
    personal_500m=("personal_aprox", "sum"),
    universidades_500m=("universidad", "sum")
)

for columna in resumen_denue.columns:
    puntos_candidatos[columna] = (
        puntos_candidatos["id_celda"]
        .map(resumen_denue[columna])
        .fillna(0)
    )

print(
    puntos_candidatos[
        ["negocios_500m", "personal_500m", "universidades_500m"]
    ].describe().round(2)
)

       negocios_500m  personal_500m  universidades_500m
count       12954.00       12954.00             12954.0
mean          397.62        2555.06                 0.6
std           485.49        3095.64                 2.6
min             0.00           0.00                 0.0
25%           107.00         589.00                 0.0
50%           311.00        1873.50                 0.0
75%           558.00        3381.38                 0.0
max          7951.00       42732.50                70.0


In [14]:
# Cargar infraestructura ciclista
ciclovias = gpd.read_file(
    "../Data/Info_infra_ciclista/infraestructura_vial_ciclista/Infraestructura ciclista total.shp"
)

# Conservar infraestructura en operación
ciclovias_activas = ciclovias[
    ciclovias["ESTADO"] == "En operacion"
].to_crs(puntos_candidatos.crs)

# Recortar infraestructura a los buffers de cada cuadrícula
intersecciones = gpd.overlay(
    ciclovias_activas[["geometry"]],
    buffers_candidatos[["id_celda", "geometry"]],
    how="intersection"
)

# Calcular longitud en metros
intersecciones["longitud_m"] = intersecciones.geometry.length

longitud_por_celda = (
    intersecciones.groupby("id_celda")["longitud_m"].sum()
)

puntos_candidatos["infra_ciclista_500m"] = (
    puntos_candidatos["id_celda"]
    .map(longitud_por_celda)
    .fillna(0)
)

print(puntos_candidatos["infra_ciclista_500m"].describe().round(2))

print(
    "\nCuadrículas sin infraestructura ciclista:",
    (puntos_candidatos["infra_ciclista_500m"] == 0).sum()
)


count    12954.00
mean       318.73
std        819.16
min          0.00
25%          0.00
50%          0.00
75%          0.00
max       9323.69
Name: infra_ciclista_500m, dtype: float64

Cuadrículas sin infraestructura ciclista: 10062


In [15]:
predictoras = [
    "poblacion_500m",
    "negocios_500m",
    "personal_500m",
    "dist_metro",
    "dist_metrobus",
    "infra_ciclista_500m",
    "estaciones_ecobici_500m",
    "universidades_500m",
    "dist_centro"
]

print("Dimensiones:", puntos_candidatos.shape)

print("\nValores faltantes:")
print(puntos_candidatos[predictoras].isna().sum())

puntos_candidatos.to_file(
    "../Data/Procesados/variables_candidatas_ECOBICI.gpkg",
    driver="GPKG"
)

Dimensiones: (12954, 12)

Valores faltantes:
poblacion_500m             0
negocios_500m              0
personal_500m              0
dist_metro                 0
dist_metrobus              0
infra_ciclista_500m        0
estaciones_ecobici_500m    0
universidades_500m         0
dist_centro                0
dtype: int64


In [16]:
entrenamiento = pd.read_csv(
    "../Data/Procesados/datos_modelo_ECOBICI.csv"
)

comparacion = pd.DataFrame({
    "max_entrenamiento": entrenamiento[predictoras].max(),
    "max_candidatas": puntos_candidatos[predictoras].max(),
    "%_candidatas_fuera_rango": [
        100 * (
            (puntos_candidatos[v] < entrenamiento[v].min()) |
            (puntos_candidatos[v] > entrenamiento[v].max())
        ).mean()
        for v in predictoras
    ]
})

print(comparacion.round(2))

                         max_entrenamiento  max_candidatas  \
poblacion_500m                    25544.00        36245.00   
negocios_500m                      7611.00         7951.00   
personal_500m                     42572.50        42732.50   
dist_metro                         2189.75        22109.50   
dist_metrobus                      4001.61        19039.18   
infra_ciclista_500m               10117.33         9323.69   
estaciones_ecobici_500m              18.00           19.00   
universidades_500m                   47.00           70.00   
dist_centro                       10964.19        34424.64   

                         %_candidatas_fuera_rango  
poblacion_500m                               2.34  
negocios_500m                               11.26  
personal_500m                               31.62  
dist_metro                                  54.31  
dist_metrobus                               41.69  
infra_ciclista_500m                          0.00  
estaciones_ecob

## Hallazgos preliminares: condiciones urbanas y oportunidades de expansión

### 1. Diferencias entre estaciones existentes y ubicaciones candidatas

El análisis de las 12,954 cuadrículas candidatas muestra diferencias importantes respecto a las zonas donde actualmente opera ECOBICI.

En particular:

- La infraestructura ciclista promedio dentro de 500 metros es de **2,837 m** para las estaciones existentes, frente a **319 m** para las cuadrículas candidatas.
- El **77.7% de las cuadrículas candidatas** no tiene infraestructura ciclista en operación registrada dentro de 500 metros, frente al 8.1% de las estaciones existentes.
- Las estaciones actuales también presentan una mayor concentración de negocios y personal ocupado en su entorno.

Estos resultados sugieren que la red actual de ECOBICI se concentra en zonas con condiciones urbanas relativamente favorables para su utilización.

Sin embargo, no es posible establecer si estas condiciones precedieron a la instalación de ECOBICI o si fueron desarrolladas como parte de su expansión.

### 2. La infraestructura urbana como posible limitación de expansión

La ausencia de infraestructura ciclista en numerosas zonas de CDMX plantea una consideración relevante: **la expansión de ECOBICI no depende exclusivamente de instalar nuevas estaciones, sino también de las condiciones urbanas que permitan utilizarlas**.

Una ubicación puede presentar una población considerable y potencial de movilidad, pero carecer de infraestructura ciclista cercana.

Esto sugiere la conveniencia de distinguir entre ubicaciones donde la expansión podría realizarse aprovechando la infraestructura existente y ubicaciones donde sería recomendable acompañarla con inversiones adicionales.

Es importante aclarar que la ausencia de infraestructura ciclista registrada no demuestra, por sí sola, que una zona sea inaccesible o insegura. Para establecerlo se necesitarían indicadores adicionales de accesibilidad, conectividad vial y seguridad.

### 3. Diferencias respecto a los datos de entrenamiento

Se identificaron diferencias considerables entre las características de las cuadrículas candidatas y las 670 estaciones utilizadas para entrenar los modelos predictivos.

En particular, el porcentaje de cuadrículas con valores fuera del rango observado en entrenamiento fue:

| Variable | Cuadrículas fuera de rango |
|---|---:|
| Estaciones ECOBICI cercanas | 93.73% |
| Distancia al centro | 63.79% |
| Distancia al Metro | 54.31% |
| Distancia al Metrobús | 41.69% |
| Personal ocupado cercano | 31.62% |

Estos resultados evidencian un desafío de generalización: nuestros modelos fueron entrenados con estaciones existentes, mientras que muchas ubicaciones candidatas presentan características poco representadas en esos datos.

Por tanto, el desempeño obtenido mediante validación espacial no necesariamente será representativo de las predicciones realizadas en toda CDMX.

### 4. Implicaciones para la metodología de expansión

Los resultados anteriores justifican incorporar las siguientes consideraciones en las etapas posteriores:

- Evaluar la plausibilidad de las predicciones en ubicaciones con características diferentes a las estaciones actuales.
- Distinguir entre expansión cercana a la red existente y expansión hacia zonas actualmente desatendidas.
- Considerar la posibilidad de instalar conjuntos de estaciones complementarias que permitan ampliar gradualmente la conectividad de ECOBICI.
- Evaluar las recomendaciones conjuntamente mediante demanda estimada, cobertura poblacional incremental y conectividad, evitando depender exclusivamente de las predicciones de demanda.

**Hallazgo preliminar:** la expansión de ECOBICI debe entenderse no solamente como un problema de selección de ubicaciones, sino también como un problema de integración con la infraestructura y la red de movilidad existentes.

Estos resultados no determinan todavía cuáles ubicaciones deben seleccionarse, pero proporcionan elementos para diseñar y justificar el modelo de optimización.

In [17]:
rangos = [0, 500, 1000, 2000, 5000, np.inf]

etiquetas = [
    "0-500 m",
    "500 m-1 km",
    "1-2 km",
    "2-5 km",
    "Más de 5 km"
]

puntos_candidatos["zona_expansion"] = pd.cut(
    puntos_candidatos["dist_ecobici_min"],
    bins=rangos,
    labels=etiquetas,
    include_lowest=True
)

resumen = puntos_candidatos.groupby(
    "zona_expansion", observed=True
).agg(
    cuadriculas=("id_celda", "count"),
    poblacion_mediana=("poblacion_500m", "median"),
    negocios_mediana=("negocios_500m", "median"),
    infra_mediana=("infra_ciclista_500m", "median")
)

print(resumen)

                cuadriculas  poblacion_mediana  negocios_mediana  \
zona_expansion                                                     
0-500 m                 814            10356.0             641.0   
500 m-1 km              392            11236.0             457.5   
1-2 km                  779            10032.0             401.0   
2-5 km                 2424             9403.5             390.0   
Más de 5 km            8545             6188.0             240.0   

                infra_mediana  
zona_expansion                 
0-500 m           1824.442913  
500 m-1 km           0.000000  
1-2 km               0.000000  
2-5 km               0.000000  
Más de 5 km          0.000000  


In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge

X_train = entrenamiento[predictoras]
y_train = entrenamiento["movimientos_diarios"]

X_candidatas = puntos_candidatos[predictoras]

# Random Forest
rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# Ridge polinómica
ridge_poli = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    Ridge(alpha=100)
)

rf.fit(X_train, y_train)
ridge_poli.fit(X_train, y_train)

print("Modelos entrenados correctamente")

Modelos entrenados correctamente


In [19]:
puntos_candidatos["demanda_rf"] = rf.predict(X_candidatas)
puntos_candidatos["demanda_ridge"] = ridge_poli.predict(X_candidatas)

print(
    puntos_candidatos[
        ["demanda_rf", "demanda_ridge"]
    ].describe().round(2)
)

print("\nPredicciones negativas:")
print(
    (puntos_candidatos[
        ["demanda_rf", "demanda_ridge"]
    ] < 0).sum()
)

print("\nDemanda promedio por zona:")
print(
    puntos_candidatos.groupby(
        "zona_expansion", observed=True
    )[["demanda_rf", "demanda_ridge"]]
    .mean()
    .round(2)
)

       demanda_rf  demanda_ridge
count    12954.00       12954.00
mean        93.58         926.16
std         22.66        1577.15
min         34.95        -231.08
25%         83.90          95.19
50%         87.58         183.60
75%         95.63        1006.27
max        373.17       14301.29

Predicciones negativas:
demanda_rf         0
demanda_ridge    801
dtype: int64

Demanda promedio por zona:
                demanda_rf  demanda_ridge
zona_expansion                           
0-500 m             124.29         130.20
500 m-1 km          104.78         110.96
1-2 km               97.27         103.51
2-5 km               95.40         123.17
Más de 5 km          89.29        1342.17


In [20]:
variables_revision = [
    "dist_centro",
    "dist_metro",
    "dist_metrobus",
    "estaciones_ecobici_500m"
]

for v in variables_revision:
    puntos_candidatos[f"fuera_{v}"] = (
        (puntos_candidatos[v] < entrenamiento[v].min()) |
        (puntos_candidatos[v] > entrenamiento[v].max())
    )

resumen_aplicabilidad = puntos_candidatos.groupby(
    "zona_expansion", observed=True
).agg(
    cuadriculas=("id_celda", "count"),
    pct_fuera_centro=("fuera_dist_centro", "mean"),
    pct_fuera_metro=("fuera_dist_metro", "mean"),
    pct_fuera_metrobus=("fuera_dist_metrobus", "mean"),
    pct_fuera_ecobici=("fuera_estaciones_ecobici_500m", "mean")
)

columnas_pct = resumen_aplicabilidad.columns.drop("cuadriculas")
resumen_aplicabilidad[columnas_pct] *= 100

print(resumen_aplicabilidad.round(1))

                cuadriculas  pct_fuera_centro  pct_fuera_metro  \
zona_expansion                                                   
0-500 m                 814               1.1              1.0   
500 m-1 km              392               8.7              6.4   
1-2 km                  779              17.6              8.3   
2-5 km                 2424              39.1             31.9   
Más de 5 km            8545              83.5             72.1   

                pct_fuera_metrobus  pct_fuera_ecobici  
zona_expansion                                         
0-500 m                        1.1                0.2  
500 m-1 km                     1.8              100.0  
1-2 km                         2.2              100.0  
2-5 km                         9.9              100.0  
Más de 5 km                   60.0              100.0  


## Hallazgos: aplicabilidad de los modelos predictivos a nuevas ubicaciones

### 1. Predicciones iniciales de demanda

Se aplicaron los modelos Random Forest y Ridge polinómica, entrenados con 670 estaciones existentes, a las 12,954 cuadrículas candidatas de CDMX.

Los resultados evidenciaron diferencias importantes entre ambos modelos:

| Indicador | Random Forest | Ridge polinómica |
|---|---:|---:|
| Demanda promedio predicha | 93.58 | 926.16 |
| Mediana | 87.58 | 183.60 |
| Predicción mínima | 34.95 | −231.08 |
| Predicción máxima | 373.17 | 14,301.29 |
| Predicciones negativas | 0 | 801 |

La regresión Ridge polinómica presentó valores negativos y estimaciones excesivamente elevadas, incompatibles con una interpretación razonable de movimientos diarios por estación.

Este comportamiento se atribuye probablemente a la extrapolación de términos cuadráticos e interacciones hacia valores de las variables explicativas poco representados durante el entrenamiento.

Por su parte, Random Forest produjo predicciones numéricamente más plausibles y menos dispersas, aunque esto no garantiza su precisión en ubicaciones nuevas.

**Decisión metodológica:** no utilizar Ridge polinómica, en su configuración actual, para estimar demanda en toda CDMX. Se conservará Random Forest como modelo predictivo provisional, sujeto a restricciones de aplicabilidad y análisis de sensibilidad.

### 2. Diferencias geográficas entre entrenamiento y predicción

Se clasificaron las cuadrículas candidatas según su distancia a la estación ECOBICI existente más cercana.

Posteriormente, se identificó qué proporción de cuadrículas presentaba valores fuera de los rangos observados durante el entrenamiento para cuatro variables relevantes.

| Distancia a ECOBICI | Cuadrículas | Fuera de rango: centro | Fuera de rango: Metro | Fuera de rango: Metrobús | Fuera de rango: ECOBICI cercana |
|---|---:|---:|---:|---:|---:|
| 0–500 m | 814 | 1.1% | 1.0% | 1.1% | 0.2% |
| 500 m–1 km | 392 | 8.7% | 6.4% | 1.8% | 100% |
| 1–2 km | 779 | 17.6% | 8.3% | 2.2% | 100% |
| 2–5 km | 2,424 | 39.1% | 31.9% | 9.9% | 100% |
| Más de 5 km | 8,545 | 83.5% | 72.1% | 60.0% | 100% |

Los resultados muestran que, conforme aumenta la distancia a la red actual, una proporción creciente de las cuadrículas presenta características que exceden los rangos observados durante el entrenamiento.

En particular, las ubicaciones a menos de 500 metros de ECOBICI presentan condiciones relativamente similares a las estaciones existentes en las variables examinadas.

En cambio, las ubicaciones situadas a más de 5 km presentan diferencias sustanciales, especialmente en distancia al centro y proximidad al transporte público.

### 3. Limitaciones de generalización

El desempeño obtenido mediante validación espacial con estaciones existentes no garantiza una precisión equivalente en nuevas ubicaciones.

Se identifican dos desafíos principales:

**Sesgo de selección:** las estaciones de entrenamiento corresponden a ubicaciones donde ECOBICI ya opera. Estas no constituyen una muestra representativa de todas las posibles ubicaciones de CDMX.

**Extrapolación:** una proporción considerable de las cuadrículas candidatas presenta características no observadas durante el entrenamiento, particularmente ausencia de estaciones ECOBICI cercanas y mayores distancias al centro y al transporte público.

Aunque Random Forest produce predicciones dentro de un rango numéricamente razonable, puede asignar valores similares a ubicaciones muy diferentes cuando sus características exceden los rangos conocidos.

Por lo tanto, una predicción plausible no debe interpretarse automáticamente como una predicción confiable.

### 4. Implicaciones para la expansión de ECOBICI

Los resultados sugieren distinguir dos estrategias de expansión:

**Expansión conectada:** incorporación de estaciones próximas a la red existente, donde las condiciones urbanas presentan mayor similitud con los datos de entrenamiento.

**Expansión hacia nuevas zonas:** identificación de conjuntos de ubicaciones actualmente alejadas de ECOBICI, donde podría requerirse instalar varias estaciones complementarias y desarrollar infraestructura ciclista adicional.

En el primer escenario, la demanda estimada mediante Random Forest puede utilizarse como criterio orientativo, reconociendo sus limitaciones.

En el segundo escenario, existe mayor incertidumbre predictiva. Por ello, será necesario dar especial importancia a la cobertura poblacional potencial, la disponibilidad de infraestructura y la conectividad entre las nuevas estaciones.

Estas categorías no constituyen restricciones definitivas ni demuestran que las ubicaciones alejadas tengan menor demanda potencial.

### 5. Conclusión metodológica

Los resultados muestran que **un modelo con buen desempeño en estaciones existentes no necesariamente generaliza adecuadamente a ubicaciones donde ECOBICI todavía no opera**.

La selección de nuevas estaciones no deberá basarse exclusivamente en la demanda predicha.

En consecuencia, la siguiente etapa consistirá en desarrollar un modelo de optimización que integre:

1. Demanda estimada, considerando su incertidumbre y aplicabilidad.
2. Cobertura poblacional incremental, evitando contar varias veces a los mismos habitantes.
3. Conectividad con la red existente y complementariedad entre las nuevas estaciones.

La optimización deberá permitir evaluar diferentes cantidades de estaciones nuevas (K) y comparar recomendaciones bajo distintos supuestos.

**El objetivo es construir una herramienta de apoyo a la decisión para priorizar ubicaciones de expansión, no afirmar que las predicciones representan con exactitud la demanda que generaría cada nueva estación.**